In [1]:
import os
import re
import glob
from math import radians, sin, cos, sqrt, atan2

import numpy as np
import pandas as pd
import xarray as xr

try:
    import gsw
    HAVE_GSW = True
except ImportError:
    HAVE_GSW = False
    print("gsw not installed: using UNESCO (Saunders 1981) pressure-to-depth formula")

## Settings

In [21]:
CTD_PATH = "/home/geosciences/Documents/Gurcan/data/CTD/data/"
FILE_PATTERN = "*_tps.cnv"          # e.g. "*_tps.cnv" to restrict to one processing stage
OUTFILE = "ctd_metadata.nc"

# Station grouping: a cast belongs to the previous station if it starts within
# STATION_RADIUS_KM of that station's first cast AND within STATION_MAX_GAP_H hours
# of the previous cast's end. Set USE_FILENAME_STATION = True to take the station
# number straight from the file name (e.g. "wh26st098" -> 98) instead.
STATION_RADIUS_KM = 1.0
STATION_MAX_GAP_H = 6.0
USE_FILENAME_STATION = False

# Global attributes written to the file -- fill in the blanks
CRUISE_ATTRS = {
    "title": "CTD cast metadata, cruise WHIRLS",
    "cruise": "WHIRLS",
    "platform": "R/V Marion Dufresne II",
    "platform_code": "MDII",
    "instrument": "Sea-Bird SBE 911plus CTD (SBE 9 + SBE 11plus deck unit)",
    "principal_investigator": "",
    "institution": "Ecole Normale Superieure",
    "creator_name": "Emma Gurcan",
    "creator_email": "emma.gurcan@lmd.ipsl.fr",
    "project": "",
    "comment": ("start/bottom/end events derived from CTD scans: first good scan, "
                "scan of maximum pressure, last good scan."),
    "Conventions": "CF-1.8, ACDD-1.3",
}

BAD_FLAG = -9.990e-29

## Helper functions

In [22]:
def haversine(lon1, lat1, lon2, lat2):
    # great-circle distance in km
    R = 6371.0
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def pressure_to_depth(p_dbar, lat):
    # depth (m, positive down) from pressure (dbar)
    if HAVE_GSW:
        return float(-gsw.z_from_p(p_dbar, lat))
    # UNESCO 1983 / Saunders & Fofonoff
    x = sin(radians(lat)) ** 2
    g = 9.780318 * (1.0 + (5.2788e-3 + 2.36e-5 * x) * x) + 1.092e-6 * p_dbar
    d = (((-1.82e-15 * p_dbar + 2.279e-10) * p_dbar - 2.2512e-5) * p_dbar + 9.72659) * p_dbar
    return float(d / g)


def cast_type(max_pres):
    # nominal cast type (m) from maximum pressure
    if max_pres < 600:
        return 300
    elif max_pres < 1500:
        return 1200
    return 5000


def parse_cnv(path):
    # Read a Sea-Bird .cnv: returns (header_lines, DataFrame with short column names).
    # Bad-flag values become NaN.
    header, names = [], {}
    with open(path, "r", errors="replace") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("*END*"):
                break
            header.append(line)
            m = re.match(r"#\s*name\s+(\d+)\s*=\s*([^:]+):", line)
            if m:
                names[int(m.group(1))] = m.group(2).strip()
        cols = [names[i] for i in sorted(names)]
        df = pd.read_csv(fh, sep=r"\s+", header=None, names=cols, engine="python")
    df = df.mask((df - BAD_FLAG).abs() < 1e-31)
    return header, df


def parse_sensors(header):
    # Serial numbers and calibration dates from the XML sensor block in the header.
    # Returns a dict like {"temperature_1_serial": "6978", "temperature_1_caldate": "10-Jan-25", ...}
    xml = "\n".join(l.lstrip("#").strip() for l in header if l.startswith("#"))
    out = {}
    blocks = re.findall(r"<!--\s*(.*?)\s*-->\s*<(\w+)[^>]*>(.*?)</\2>", xml, flags=re.S)
    for comment, tag, body in blocks:
        name = comment.split(",", 1)[1].strip() if "," in comment else comment
        key = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        if not key.endswith(("_2",)):
            key = key + "_1"
        sn = re.search(r"<SerialNumber>(.*?)</SerialNumber>", body)
        cd = re.search(r"<CalibrationDate>(.*?)</CalibrationDate>", body)
        if sn:
            out[f"{key}_serial"] = sn.group(1).strip()
        if cd:
            out[f"{key}_caldate"] = cd.group(1).strip()
    return out


def parse_header_info(header):
    # Cruise / ship / NMEA start info written by Seasave
    info = {}
    for l in header:
        m = re.match(r"\*\*\s*(\w[\w ]*?)\s*:\s*(.*)", l)
        if m and m.group(2).strip():
            info[m.group(1).strip().lower()] = m.group(2).strip()
        m = re.match(r"\*\s*NMEA (Latitude|Longitude)\s*=\s*(\d+)\s+([\d.]+)\s*([NSEW])", l)
        if m:
            dd = float(m.group(2)) + float(m.group(3)) / 60.0
            if m.group(4) in "SW":
                dd = -dd
            info["nmea_" + m.group(1).lower()] = dd
    return info


def scan_times(df, year_hint=None):
    # UTC timestamps for each scan. Prefer timeQ (NMEA seconds since 2000-01-01),
    # fall back to timeJ (Julian day of year, needs the year).
    if "timeQ" in df:
        return pd.to_datetime(df["timeQ"], unit="s", origin=pd.Timestamp("2000-01-01"))
    if "timeJ" in df:
        if year_hint is None:
            raise ValueError("timeJ only: give year_hint")
        return pd.Timestamp(f"{year_hint}-01-01") + pd.to_timedelta(df["timeJ"] - 1, unit="D")
    raise KeyError("no timeQ/timeJ column in file")


def station_from_filename(fname):
    m = re.search(r"st(\d+)", fname, flags=re.I)
    return int(m.group(1)) if m else -1

In [23]:
def extract_cast(path):
    # One CASINO-style metadata record for a .cnv file
    header, df = parse_cnv(path)
    fname = os.path.basename(path)

    pcol = "prDM" if "prDM" in df else [c for c in df if c.startswith("pr")][0]
    good = df[pcol].notna() & df["latitude"].notna() & df["longitude"].notna()
    if "timeQ" in df:
        good &= df["timeQ"].notna()
    d = df[good].reset_index(drop=True)
    if d.empty:
        raise ValueError(f"{fname}: no good scans")

    t = scan_times(d)
    p = d[pcol].to_numpy()
    i_start, i_bot, i_end = 0, int(np.nanargmax(p)), len(d) - 1

    rec = {"file": fname, "filename_station": station_from_filename(fname)}
    for ev, i in (("start", i_start), ("bottom", i_bot), ("end", i_end)):
        rec[f"time_{ev}"] = t.iloc[i]
        rec[f"lat_{ev}"] = float(d["latitude"].iloc[i])
        rec[f"lon_{ev}"] = float(d["longitude"].iloc[i])

    rec["max_pressure"] = float(p[i_bot])
    rec["max_depth"] = pressure_to_depth(p[i_bot], rec["lat_bottom"])
    rec["cast_type"] = cast_type(p[i_bot])
    rec["duration_min"] = (rec["time_end"] - rec["time_start"]).total_seconds() / 60.0
    rec["drift_km"] = haversine(rec["lon_start"], rec["lat_start"], rec["lon_end"], rec["lat_end"])
    rec["n_scans"] = len(d)

    # Altimeter (only if exported in DatCnv; column usually "altM")
    alt_col = next((c for c in df if c.lower().startswith("alt")), None)
    if alt_col is not None:
        win = d[alt_col].iloc[max(0, i_bot - 10): i_bot + 11]
        alt = float(win[(win > 0) & (win < 99)].min()) if ((win > 0) & (win < 99)).any() else np.nan
        rec["altimeter_at_bottom"] = alt
        rec["seafloor_depth"] = rec["max_depth"] + alt if np.isfinite(alt) else np.nan
    else:
        rec["altimeter_at_bottom"] = np.nan
        rec["seafloor_depth"] = np.nan

    rec["header"] = parse_header_info(header)
    rec["sensors"] = parse_sensors(header)
    return rec


def assign_stations(casts, radius_km=1.0, max_gap_h=6.0):
    # Group consecutive casts into stations and label them 1a, 1b, 2a, ...
    # Compares against the FIRST cast of the current station (not the previous one),
    # so slow drift over several casts doesn't split a station.
    station = 0
    ref = None
    letters = {}
    prev_end = None
    for c in casts:
        new = True
        if ref is not None:
            dist = haversine(ref["lon_start"], ref["lat_start"], c["lon_start"], c["lat_start"])
            gap_h = (c["time_start"] - prev_end).total_seconds() / 3600.0
            new = dist > radius_km or gap_h > max_gap_h
        if new:
            station += 1
            ref = c
            letters[station] = 0
        letters[station] += 1
        c["station_number"] = station
        c["station_id"] = f"{station}{chr(96 + letters[station])}"
        prev_end = c["time_end"]
    return casts


def stations_from_filename(casts):
    letters = {}
    for c in casts:
        s = c["filename_station"]
        letters[s] = letters.get(s, 0) + 1
        c["station_number"] = s
        c["station_id"] = f"{s}{chr(96 + letters[s])}"
    return casts

In [24]:
VAR_ATTRS = {
    "file":              dict(long_name="source .cnv file name"),
    "cast_number":       dict(long_name="cast number (chronological)"),
    "station_number":    dict(long_name="station number"),
    "station_id":        dict(long_name="station and cast label (station number + letter)"),
    "filename_station":  dict(long_name="station number parsed from file name (-1 if none)"),
    "max_pressure":      dict(long_name="maximum pressure of cast", units="dbar",
                              standard_name="sea_water_pressure"),
    "max_depth":         dict(long_name="maximum depth of CTD", units="m", positive="down",
                              standard_name="depth"),
    "cast_type":         dict(long_name="nominal cast type (300/1200/5000 m)", units="m"),
    "seafloor_depth":    dict(long_name="seafloor depth from CTD depth + altimeter", units="m",
                              positive="down", standard_name="sea_floor_depth_below_sea_surface"),
    "altimeter_at_bottom": dict(long_name="altimeter range at bottom of cast", units="m"),
    "duration_min":      dict(long_name="cast duration (start to end)", units="minutes"),
    "drift_km":          dict(long_name="ship drift between start and end of cast", units="km"),
    "n_scans":           dict(long_name="number of good scans in file"),
}
for ev, desc in (("start", "CTD in water (first good scan)"),
                 ("bottom", "maximum pressure reached"),
                 ("end", "CTD out of water (last good scan)")):
    VAR_ATTRS[f"time_{ev}"] = dict(long_name=f"UTC time: {desc}", standard_name="time")
    VAR_ATTRS[f"lat_{ev}"] = dict(long_name=f"latitude: {desc}", units="degrees_north",
                                  standard_name="latitude")
    VAR_ATTRS[f"lon_{ev}"] = dict(long_name=f"longitude: {desc}", units="degrees_east",
                                  standard_name="longitude")


def build_dataset(casts):
    n = len(casts)
    scalar_vars = ["filename_station", "station_number", "max_pressure", "max_depth",
                   "cast_type", "seafloor_depth", "altimeter_at_bottom", "duration_min",
                   "drift_km", "n_scans"]
    for ev in ("start", "bottom", "end"):
        scalar_vars += [f"lat_{ev}", f"lon_{ev}"]

    data_vars = {v: ("cast", np.array([c[v] for c in casts])) for v in scalar_vars}
    for ev in ("start", "bottom", "end"):
        data_vars[f"time_{ev}"] = ("cast", pd.to_datetime([c[f"time_{ev}"] for c in casts]).values)
    data_vars["file"] = ("cast", np.array([c["file"] for c in casts], dtype=object))

    # sensor serial numbers / cal dates as per-cast string variables (they can change mid-cruise)
    sensor_keys = sorted({k for c in casts for k in c["sensors"]})
    for k in sensor_keys:
        data_vars[f"sensor_{k}"] = ("cast", np.array([c["sensors"].get(k, "") for c in casts],
                                                      dtype=object))

    ds = xr.Dataset(
        data_vars=data_vars,
        coords=dict(
            cast=("cast", np.arange(1, n + 1)),
            station_id=("cast", np.array([c["station_id"] for c in casts], dtype=object)),
        ),
    )

    ds["cast"].attrs = VAR_ATTRS["cast_number"]
    for v, a in VAR_ATTRS.items():
        if v in ds:
            ds[v].attrs.update(a)
    for k in sensor_keys:
        what = "serial number" if k.endswith("_serial") else "calibration date"
        ds[f"sensor_{k}"].attrs["long_name"] = f"{what}: {k.rsplit('_', 1)[0].replace('_', ' ')}"

    # global attributes
    attrs = dict(CRUISE_ATTRS)
    hdr = casts[0]["header"]
    attrs.setdefault("cruise", hdr.get("cruise", ""))
    attrs["time_coverage_start"] = str(min(c["time_start"] for c in casts))
    attrs["time_coverage_end"] = str(max(c["time_end"] for c in casts))
    lats = [c[f"lat_{e}"] for c in casts for e in ("start", "end")]
    lons = [c[f"lon_{e}"] for c in casts for e in ("start", "end")]
    attrs.update(geospatial_lat_min=float(np.min(lats)), geospatial_lat_max=float(np.max(lats)),
                 geospatial_lon_min=float(np.min(lons)), geospatial_lon_max=float(np.max(lons)))
    attrs["date_created"] = pd.Timestamp.now("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    attrs["number_of_casts"] = n
    attrs["number_of_stations"] = int(len(set(c["station_number"] for c in casts)))
    attrs["pressure_to_depth"] = "gsw.z_from_p (TEOS-10)" if HAVE_GSW else "UNESCO 1983 (Saunders & Fofonoff)"
    ds.attrs = attrs
    return ds


def encoding_for(ds):
    enc = {}
    for v in ds.data_vars:
        if v.startswith("time_"):
            enc[v] = {"units": "seconds since 1970-01-01 00:00:00", "calendar": "standard",
                      "dtype": "float64"}
        elif ds[v].dtype.kind == "f":
            enc[v] = {"_FillValue": np.nan}
        elif ds[v].dtype == object:
            enc[v] = {"dtype": str}
    enc["station_id"] = {"dtype": str}
    return enc

In [25]:
os.path.join(CTD_PATH, FILE_PATTERN)

'/home/geosciences/Documents/Gurcan/data/CTD/data/*_tps.cnv'

## Read all casts

In [27]:
files = sorted(glob.glob(os.path.join(CTD_PATH, FILE_PATTERN)))
print(f"{len(files)} files found")

casts, failed = [], []
for f in files:
    try:
        casts.append(extract_cast(f))
    except Exception as e:
        failed.append((os.path.basename(f), str(e)))

# order chronologically (not by file name) before numbering
casts.sort(key=lambda c: c["time_start"])
if USE_FILENAME_STATION:
    casts = stations_from_filename(casts)
else:
    casts = assign_stations(casts, STATION_RADIUS_KM, STATION_MAX_GAP_H)

print(f"{len(casts)} casts read, {len(failed)} failed")
for name, err in failed:
    print("  FAILED", name, "->", err)

105 files found
105 casts read, 0 failed


## Check before writing

In [28]:
summary = pd.DataFrame([{k: c[k] for k in (
    "station_id", "file", "time_start", "time_bottom", "time_end",
    "lat_start", "lon_start", "max_pressure", "max_depth", "cast_type",
    "duration_min", "drift_km")} for c in casts])
summary.index = np.arange(1, len(summary) + 1)
pd.set_option("display.width", 200)
summary.round(3)

/tmp/ipykernel_7861/1914922507.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  summary.round(3)


,station_id,file,time_start,time_bottom,time_end,lat_start,lon_start,max_pressure,max_depth,cast_type,duration_min,drift_km
1,1a,wh26st001_tps.cnv,2026-06-29 20:41:23,2026-06-29 21:07:09,2026-06-29 21:07:22,-34.500,16.637,326.211,323.614,300,25.983,0.087
2,1b,wh26st002_tps.cnv,2026-06-29 22:02:58,2026-06-29 22:36:26,2026-06-29 23:30:10,-34.500,16.638,1200.327,1188.282,1200,87.200,0.510
3,2a,wh26st003_tps.cnv,2026-06-30 10:31:17,2026-06-30 12:23:27,2026-06-30 14:05:50,-34.500,15.428,4473.147,4394.954,5000,214.550,1.188
4,3a,wh26st004_tps.cnv,2026-07-01 17:14:06,2026-07-01 17:23:03,2026-07-01 17:35:53,-37.402,11.636,202.230,200.629,300,21.783,0.002
5,3b,wh26st005_tps.cnv,2026-07-01 18:32:11,2026-07-01 19:04:06,2026-07-01 19:47:19,-37.402,11.636,1200.850,1188.496,1200,75.133,0.068
...,...,...,...,...,...,...,...,...,...,...,...,...
101,85b,wh26st101_tps.cnv,2026-07-26 13:43:51,2026-07-26 14:15:32,2026-07-26 15:06:07,-35.547,15.567,1201.394,1189.228,1200,82.267,1.512
102,86a,wh26st102_tps.cnv,2026-07-26 16:24:14,2026-07-26 16:59:29,2026-07-26 17:52:13,-35.473,15.708,1206.225,1194.005,1200,87.983,1.617
103,87a,wh26st103_tps.cnv,2026-07-26 19:04:14,2026-07-26 19:33:59,2026-07-26 20:16:50,-35.393,15.844,1201.099,1188.953,1200,72.600,0.312
104,88a,wh26st104_tps.cnv,2026-07-26 21:27:14,2026-07-26 21:57:36,2026-07-26 22:39:11,-35.316,15.987,1201.131,1188.992,1200,71.950,0.469


In [29]:
# sanity checks: header NMEA position vs first scan, odd casts
for c in casts:
    h = c["header"]
    if "nmea_latitude" in h:
        dkm = haversine(h["nmea_longitude"], h["nmea_latitude"], c["lon_start"], c["lat_start"])
        if dkm > 1:
            print(f"{c['file']}: first scan is {dkm:.1f} km from header NMEA position")
    if c["drift_km"] > 2:
        print(f"{c['file']}: ship drifted {c['drift_km']:.1f} km during cast")
    if c["max_pressure"] < 20:
        print(f"{c['file']}: max pressure only {c['max_pressure']:.1f} dbar (aborted cast?)")

wh26st044_tps.cnv: ship drifted 4.5 km during cast
wh26st045_tps.cnv: ship drifted 3.2 km during cast
wh26st046_tps.cnv: ship drifted 5.1 km during cast
wh26st047_tps.cnv: ship drifted 2.8 km during cast
wh26st048_tps.cnv: ship drifted 3.0 km during cast
wh26st049_tps.cnv: ship drifted 2.4 km during cast
wh26st051_tps.cnv: ship drifted 3.9 km during cast
wh26st052_tps.cnv: ship drifted 2.6 km during cast
wh26st053_tps.cnv: ship drifted 4.0 km during cast
wh26st055_tps.cnv: ship drifted 2.8 km during cast
wh26st058_tps.cnv: ship drifted 4.0 km during cast
wh26st059_tps.cnv: ship drifted 2.9 km during cast
wh26st060_tps.cnv: ship drifted 3.1 km during cast
wh26st061_tps.cnv: ship drifted 2.9 km during cast
wh26st062_tps.cnv: ship drifted 2.9 km during cast
wh26st063_tps.cnv: ship drifted 2.1 km during cast
wh26st065_tps.cnv: ship drifted 2.9 km during cast
wh26st067_tps.cnv: ship drifted 4.0 km during cast
wh26st068_tps.cnv: ship drifted 3.4 km during cast
wh26st069_tps.cnv: ship drifted

## Write NetCDF

In [30]:
ds = build_dataset(casts)
ds.to_netcdf(OUTFILE, encoding=encoding_for(ds))
print(f"Saved: {OUTFILE}")
ds

Saved: ctd_metadata.nc


<xarray.Dataset> Size: 35kB
Dimensions:                                           (cast: 105)
Coordinates:
  * cast                                              (cast) int64 840B 1 ......
    station_id                                        (cast) object 840B '1a'...
Data variables: (12/40)
    filename_station                                  (cast) int64 840B 1 ......
    station_number                                    (cast) int64 840B 1 ... 89
    max_pressure                                      (cast) float64 840B 326...
    max_depth                                         (cast) float64 840B 323...
    cast_type                                         (cast) int64 840B 300 ....
    seafloor_depth                                    (cast) float64 840B nan...
    ...                                                ...
    sensor_temperature_1_caldate                      (cast) object 840B '10-...
    sensor_temperature_1_serial                       (cast) object 840B '697...
    sensor_temperature_2_caldate                      (cast) object 840B '08-...
    sensor_temperature_2_serial                       (cast) object 840B '701...
    sensor_transmissometer_wet_labs_c_star_1_caldate  (cast) object 840B '202...
    sensor_transmissometer_wet_labs_c_star_1_serial   (cast) object 840B '111...
Attributes: (12/22)
    title:                   CTD cast metadata, cruise WHIRLS
    cruise:                  WHIRLS
    platform:                R/V Marion Dufresne II
    platform_code:           MDII
    instrument:              Sea-Bird SBE 911plus CTD (SBE 9 + SBE 11plus dec...
    principal_investigator:  
    ...                      ...
    geospatial_lon_min:      9.5263
    geospatial_lon_max:      16.71584
    date_created:            2026-10-16T08:41:40Z
    number_of_casts:         105
    number_of_stations:      89
    pressure_to_depth:       gsw.z_from_p (TEOS-10)

In [31]:
# re-open to confirm it reads back correctly
with xr.open_dataset(OUTFILE) as check:
    print(check[["station_id", "time_start", "max_pressure", "cast_type"]].to_dataframe().head(10))

     station_id          time_start  max_pressure  cast_type
cast                                                        
1            1a 2026-06-29 20:41:23       326.211        300
2            1b 2026-06-29 22:02:58      1200.327       1200
3            2a 2026-06-30 10:31:17      4473.147       5000
4            3a 2026-07-01 17:14:06       202.230        300
5            3b 2026-07-01 18:32:11      1200.850       1200
6            4a 2026-07-02 05:35:26       250.999        300
7            4b 2026-07-02 06:46:07      1204.976       1200
8            5a 2026-07-02 20:48:55       301.595        300
9            5b 2026-07-02 21:49:36      1201.748       1200
10           6a 2026-07-03 07:05:52      1202.731       1200


## Optional: merge seafloor depth / event times from a CASINO export

The column names below are placeholders -- adjust them to your CASINO export once you
have it. Matching is by nearest CASINO event time within `tol` of each cast's bottom time.

In [ ]:
CASINO_FILE = None   # e.g. "casino_events.csv"

if CASINO_FILE:
    cas = pd.read_csv(CASINO_FILE)
    cas["time"] = pd.to_datetime(cas["Date"] + " " + cas["Heure"], dayfirst=True)  # adjust
    cas = cas.sort_values("time")
    bottoms = pd.DataFrame({"cast": ds["cast"].values,
                            "time_bottom": pd.to_datetime(ds["time_bottom"].values)})
    m = pd.merge_asof(bottoms.sort_values("time_bottom"), cas,
                      left_on="time_bottom", right_on="time",
                      direction="nearest", tolerance=pd.Timedelta("15min"))
    m = m.set_index("cast").reindex(ds["cast"].values)
    ds["casino_water_depth"] = ("cast", m["Sonde"].astype(float).values)   # adjust column name
    ds["casino_water_depth"].attrs = dict(long_name="water depth from ship echosounder (CASINO)",
                                          units="m", positive="down",
                                          standard_name="sea_floor_depth_below_sea_surface")
    ds.to_netcdf(OUTFILE, encoding=encoding_for(ds))
    print("CASINO depths merged and file rewritten")